# Ch 11　Human-in-the-loop、interrupts 與 time travel

> **本 notebook 完全不需要 API key。** interrupt / resume / time travel 都是純機制，最適合動手玩。
>
> （`grandalf` 是 `draw_ascii()` 渲染拓樸圖用的；不裝也不影響主程式。）

In [ ]:
!uv pip install -q langgraph grandalf

## 11.1–11.2　暫停（interrupt）與回復（Command resume）

在「送出回覆」前插一個審核關卡：人核准才送，否則退回。

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command   # interrupt 暫停、Command 回復

class State(TypedDict):
    draft: str
    approved: bool

def make_draft(state): return {"draft": "親愛的客戶，已收到您的問題，我們會盡快協助您。"}

def approval(state):
    # 在這裡按下暫停鍵，把草稿丟給人審核。圖會存好狀態、無限期等待。
    ok = interrupt({"draft": state["draft"], "action": "這封回覆可以送嗎？"})
    return {"approved": ok}   # resume 時 Command(resume=...) 的值會回到 ok

def send(state):
    return {"draft": state["draft"] + ("（已送出）" if state["approved"] else "（已退回改寫）")}

graph = (StateGraph(State)
         .add_node("make_draft", make_draft).add_node("approval", approval).add_node("send", send)
         .add_edge(START, "make_draft").add_edge("make_draft", "approval")
         .add_edge("approval", "send").add_edge("send", END)
         .compile(checkpointer=InMemorySaver()))   # interrupt 一定要有 checkpointer！

In [ ]:
# interrupt 需要 thread_id（持久游標）。第一次 invoke 會停在 approval。
config = {"configurable": {"thread_id": "mail-1"}}
res = graph.invoke({"draft": "", "approved": False}, config, version="v2")

# v2 下回傳 GraphOutput；暫停的 payload 在 .interrupts
print("⏸  圖暫停了，等待審核：")
print("   ", res.interrupts[0].value)

In [ ]:
# 拿到人的決定後，用「同一個 thread_id」+ Command(resume=...) 回復。
# resume 的值會成為節點裡 interrupt() 的回傳值。
final = graph.invoke(Command(resume=True), config, version="v2")   # True = 核准
print("✅ 核准後的最終 state：", final.value)

# 換一個 thread 重跑、這次退回
config2 = {"configurable": {"thread_id": "mail-2"}}
graph.invoke({"draft": "", "approved": False}, config2, version="v2")
print("❌ 退回後：", graph.invoke(Command(resume=False), config2, version="v2").value)

## 11.3　最大的坑：resume 會「從節點開頭重跑」

interrupt 之前的程式碼，resume 時會**再執行一次**。下面用一個計數器親眼證明。

In [ ]:
calls = {"n": 0}

class S(TypedDict):
    approved: bool

def approval_with_sidefx(state):
    calls["n"] += 1
    print(f"  ⚠️ approval 節點第 {calls['n']} 次執行（這行在 interrupt 之前）")
    ok = interrupt("approve?")
    return {"approved": ok}

g = (StateGraph(S).add_node("approval", approval_with_sidefx)
     .add_edge(START, "approval").add_edge("approval", END)
     .compile(checkpointer=InMemorySaver()))
c = {"configurable": {"thread_id": "x1"}}
g.invoke({"approved": False}, c, version="v2")    # 印第 1 次
g.invoke(Command(resume=True), c, version="v2")   # resume → 印第 2 次！
print("=> interrupt 前的副作用會重複。所以要嘛冪等、要嘛移到 interrupt 之後。")

## 11.4.5　Review-and-edit：人不只能核准，還能直接改寫草稿

approve/reject 只是陽春版。production 更常見的是讓人**直接改寫** agent 草稿再放行——下面用 dict / True / False 三種 resume 值區分意圖。

In [ ]:
class MailS(TypedDict):
    draft: str
    approved: bool

def make_draft(s):
    return {"draft": "親愛的客戶，已收到您的問題，我們會盡快協助您。"}

def review(s):
    decision = interrupt({"draft": s["draft"], "action": "approve | edit | reject"})
    # 三種決策：dict (改寫) / True (核准原稿) / False (退回)
    if isinstance(decision, dict) and decision.get("action") == "edit":
        return {"draft": decision["new_draft"], "approved": True}
    if decision is True:
        return {"approved": True}
    return {"approved": False}

def send(s):
    tag = "（已送出）" if s["approved"] else "（已退回改寫）"
    return {"draft": s["draft"] + tag}

mg = (StateGraph(MailS)
      .add_node("make_draft", make_draft).add_node("review", review).add_node("send", send)
      .add_edge(START, "make_draft").add_edge("make_draft", "review")
      .add_edge("review", "send").add_edge("send", END)
      .compile(checkpointer=InMemorySaver()))

# 三種情境，看 resume 帶不同值會怎樣
for tid, action, resume in [
    ("r-approve", "approve", True),
    ("r-edit",    "edit",    {"action": "edit", "new_draft": "親愛的 Kevin，已收到您的回報，我們今日內回覆。"}),
    ("r-reject",  "reject",  False),
]:
    cfg = {"configurable": {"thread_id": tid}}
    mg.invoke({"draft": "", "approved": False}, cfg, version="v2")
    final = mg.invoke(Command(resume=resume), cfg, version="v2")
    print(f"[{action:7}] →", final.value)
# => 同一個 review 節點，靠 resume 值區分意圖；這是 production HITL 的常見模式。

## 11.6　Time travel：fork 出另一條路

在過去某個 checkpoint 改寫狀態、分岔出新分支——原始歷史完整保留。

In [ ]:
from typing_extensions import NotRequired

class JokeState(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]

def gen_topic(state): return {"topic": "襪子"}
def write_joke(state): return {"joke": f"為什麼{state['topic']}會不見？因為它私奔了！"}

jg = (StateGraph(JokeState).add_node("gen_topic", gen_topic).add_node("write_joke", write_joke)
      .add_edge(START, "gen_topic").add_edge("gen_topic", "write_joke").add_edge("write_joke", END)
      .compile(checkpointer=InMemorySaver()))

cfg = {"configurable": {"thread_id": "joke-1"}}
print("原始路線：", jg.invoke({}, cfg))

# 找到「write_joke 還沒跑」的 checkpoint，改 topic，分岔重跑
history = list(jg.get_state_history(cfg))
before_joke = next(s for s in history if s.next == ("write_joke",))
fork_cfg = jg.update_state(before_joke.config, values={"topic": "雞"})   # 建立新分支
print("fork 路線：", jg.invoke(None, fork_cfg))   # write_joke 用新 topic 重跑
print("=> 同一個起點，分岔出『雞』的版本；原始『襪子』歷史仍在。")

## 拓樸圖：眼見為憑

把我們本章建的 review 圖實際拓樸印出來，確認你腦中的形狀對得上程式碼。

In [ ]:
# draw_ascii() 需要 grandalf；draw_mermaid() 不必，回傳 mermaid 原始碼可貼進任何支援 mermaid 的工具
print("--- ASCII ---")
print(mg.get_graph().draw_ascii())
print("\n--- Mermaid 原始碼 ---")
print(mg.get_graph().draw_mermaid())